# End-to-End Book Recommendation: ALS & LightFM

This notebook demonstrates the full pipeline locally using **dummy data**:
1. Generate synthetic interaction data
2. Preprocess (weighting, k-core filtering)
3. Train ALS model
4. Train LightFM model
5. Evaluate both (P@10, R@10, NDCG@10, HR@10)
6. Visualize results

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded.')

## 1. Generate Dummy Data

We simulate an e-commerce bookstore with:
- 2,000 users, 500 items
- Events: page_view, wishlist, add_to_cart, purchase
- User features: city, gender, age_bucket
- Item features: category

In [ ]:
N_USERS = 2000
N_ITEMS = 500
N_EVENTS = 50000

# Generate interactions
user_ids = [f'U{i:04d}' for i in range(N_USERS)]
item_ids = [f'I{i:04d}' for i in range(N_ITEMS)]
events = np.random.choice(['page_view', 'wishlist', 'add_to_cart', 'purchase'],
                          size=N_EVENTS, p=[0.52, 0.24, 0.18, 0.06])

# Power-law distribution for users and items (some are more active)
user_probs = np.random.pareto(1.5, N_USERS) + 1
user_probs /= user_probs.sum()
item_probs = np.random.pareto(1.2, N_ITEMS) + 1
item_probs /= item_probs.sum()

log_df = pd.DataFrame({
    'user_id': np.random.choice(user_ids, size=N_EVENTS, p=user_probs),
    'item_id': np.random.choice(item_ids, size=N_EVENTS, p=item_probs),
    'event_type': events
})

print(f'Generated {len(log_df):,} events')
print(f'Users: {log_df["user_id"].nunique()}, Items: {log_df["item_id"].nunique()}')
print(f'\nEvent distribution:')
print(log_df['event_type'].value_counts())

In [ ]:
# Generate user features
cities = ['Jakarta', 'Surabaya', 'Bandung', 'Medan', 'Semarang',
          'Makassar', 'Palembang', 'Tangerang', 'Depok', 'Bekasi']
genders = ['male', 'female']
age_buckets = ['teen', 'young', 'adult', 'mid', 'senior']

users_df = pd.DataFrame({
    'user_id': user_ids,
    'city': np.random.choice(cities, N_USERS),
    'gender': np.random.choice(genders, N_USERS, p=[0.45, 0.55]),
    'age_bucket': np.random.choice(age_buckets, N_USERS, p=[0.1, 0.3, 0.35, 0.15, 0.1])
})

# Generate item features
categories = ['Fiction', 'Science', 'Romance', 'Business', 'Self-Help',
              'Children', 'History', 'Technology', 'Art', 'Religion']
items_df = pd.DataFrame({
    'item_id': item_ids,
    'category': np.random.choice(categories, N_ITEMS)
})

print(f'Users: {users_df.shape}')
print(f'Items: {items_df.shape}')

## 2. Preprocessing

- Apply implicit feedback weights
- Aggregate per (user, item) pair
- K-core filtering
- Train/test split (random 80/20 per user)

In [ ]:
EVENT_WEIGHTS = {'page_view': 1, 'wishlist': 3, 'add_to_cart': 5, 'purchase': 12}

log_df['weight'] = log_df['event_type'].map(EVENT_WEIGHTS)
interactions = log_df.groupby(['user_id', 'item_id'])['weight'].sum().reset_index()
print(f'Unique (user, item) pairs: {len(interactions):,}')

# K-core filtering
def apply_kcore(df, k):
    for _ in range(30):
        prev = len(df)
        uc = df.groupby('user_id').size()
        df = df[df['user_id'].isin(uc[uc >= k].index)]
        ic = df.groupby('item_id').size()
        df = df[df['item_id'].isin(ic[ic >= k].index)]
        if len(df) == prev:
            break
    return df

KCORE = 5
filtered = apply_kcore(interactions, KCORE)
n_users = filtered['user_id'].nunique()
n_items = filtered['item_id'].nunique()
density = len(filtered) / (n_users * n_items) * 100
print(f'After k-core={KCORE}: {n_users} users, {n_items} items, density={density:.2f}%')

In [ ]:
# Train/test split (random 80/20 per user)
train_list, test_list = [], []
for uid, group in filtered.groupby('user_id'):
    if len(group) < 2:
        train_list.append(group)
        continue
    tr, te = train_test_split(group, test_size=0.2, random_state=42)
    train_list.append(tr)
    test_list.append(te)

train_df = pd.concat(train_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

# Build index mappings
all_users = sorted(filtered['user_id'].unique())
all_items = sorted(filtered['item_id'].unique())
u2idx = {u: i for i, u in enumerate(all_users)}
i2idx = {s: i for i, s in enumerate(all_items)}

def build_matrix(df):
    row = df['user_id'].map(u2idx).values
    col = df['item_id'].map(i2idx).values
    val = df['weight'].values.astype(np.float32)
    return csr_matrix((val, (row, col)), shape=(len(all_users), len(all_items)))

R_train = build_matrix(train_df)
R_test = build_matrix(test_df)
print(f'Matrix shape: {R_train.shape}, Train nnz: {R_train.nnz:,}, Test nnz: {R_test.nnz:,}')

## 3. Train ALS Model

In [ ]:
import time
from implicit.als import AlternatingLeastSquares

t0 = time.time()
als_model = AlternatingLeastSquares(factors=150, iterations=15, regularization=0.1, random_state=42)
als_model.fit(R_train)
print(f'ALS trained in {time.time()-t0:.1f}s')

## 4. Train LightFM Model

In [ ]:
from lightfm import LightFM
from lightfm.data import Dataset

# Prepare feature labels
user_feature_labels = [f'city:{c}' for c in cities] + [f'gender:{g}' for g in genders] + [f'age:{a}' for a in age_buckets]
item_feature_labels = [f'cat:{c}' for c in categories]

ds = Dataset()
ds.fit(users=all_users, items=all_items,
       user_features=user_feature_labels, item_features=item_feature_labels)

lfm_train, lfm_weights = ds.build_interactions(
    ((r['user_id'], r['item_id'], r['weight']) for _, r in train_df.iterrows())
)

# Build user features
user_feat_map = users_df.set_index('user_id')[['city', 'gender', 'age_bucket']].to_dict('index')
uf_list = []
for uid in all_users:
    feats = []
    if uid in user_feat_map:
        r = user_feat_map[uid]
        feats = [f"city:{r['city']}", f"gender:{r['gender']}", f"age:{r['age_bucket']}"]
    uf_list.append((uid, feats))
uf_sparse = ds.build_user_features(uf_list, normalize=False)

# Build item features
item_cat_map = items_df.set_index('item_id')['category'].to_dict()
if_list = [(iid, [f"cat:{item_cat_map[iid]}"]) if iid in item_cat_map else (iid, []) for iid in all_items]
if_sparse = ds.build_item_features(if_list, normalize=False)

# Train
t0 = time.time()
lfm_model = LightFM(loss='warp', no_components=64, learning_rate=0.05, user_alpha=1e-6, item_alpha=1e-6)
lfm_model.fit(lfm_train, user_features=uf_sparse, item_features=if_sparse,
              sample_weight=lfm_weights, epochs=30, num_threads=4)
print(f'LightFM trained in {time.time()-t0:.1f}s')

## 5. Evaluation

In [ ]:
K = 10

def evaluate_model(get_recs_fn):
    """Evaluate a recommendation function. get_recs_fn(user_idx) -> list of item indices."""
    test_users = np.unique(R_test.nonzero()[0])
    prec, rec, ndcg_vals = [], [], []
    for u in test_users:
        relevant = set(R_test[u].indices)
        if not relevant:
            continue
        recs = get_recs_fn(u)
        hits = [1 if i in relevant else 0 for i in recs[:K]]
        nh = sum(hits)
        prec.append(nh / K)
        rec.append(nh / len(relevant))
        dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), K)))
        ndcg_vals.append(dcg / idcg if idcg else 0)
    hr = np.mean([1 if p > 0 else 0 for p in prec])
    return {'P@10': np.mean(prec)*100, 'R@10': np.mean(rec)*100,
            'NDCG@10': np.mean(ndcg_vals)*100, 'HR@10': hr*100, 'users': len(prec)}

# ALS recommendations
def als_recs(u):
    ids, _ = als_model.recommend(u, R_train[u], N=K, filter_already_liked_items=True)
    return ids

# LightFM recommendations
uid_map, _, iid_map, _ = ds.mapping()
rev_iid = {v: k for k, v in iid_map.items()}

def lfm_recs(u):
    uid = all_users[u]
    if uid not in uid_map:
        return []
    lfm_u = uid_map[uid]
    scores = lfm_model.predict(lfm_u, np.arange(len(iid_map)),
                               user_features=uf_sparse, item_features=if_sparse)
    train_items = set(R_train[u].indices)
    train_lfm = {iid_map[all_items[i]] for i in train_items if all_items[i] in iid_map}
    ranking = np.argsort(-scores)
    top = [i for i in ranking if i not in train_lfm][:K]
    return [i2idx[rev_iid[i]] for i in top if rev_iid[i] in i2idx]

print('Evaluating ALS...')
als_metrics = evaluate_model(als_recs)
print('Evaluating LightFM...')
lfm_metrics = evaluate_model(lfm_recs)

results = pd.DataFrame([als_metrics, lfm_metrics], index=['ALS', 'LightFM'])
print('\n' + '='*50)
print(results.to_string())

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: Metric comparison
metrics_names = ['P@10', 'R@10', 'NDCG@10', 'HR@10']
x = np.arange(len(metrics_names))
w = 0.35
axes[0].bar(x - w/2, [als_metrics[m] for m in metrics_names], w, label='ALS', color='#2196F3')
axes[0].bar(x + w/2, [lfm_metrics[m] for m in metrics_names], w, label='LightFM', color='#FF9800')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].set_ylabel('Score (%)')
axes[0].set_title('ALS vs LightFM')
axes[0].legend()

# Plot 2: Event distribution
event_counts = log_df['event_type'].value_counts()
colors = ['#90CAF9', '#FFE082', '#FFAB91', '#A5D6A7']
axes[1].pie(event_counts.values, labels=event_counts.index, autopct='%1.0f%%',
            colors=colors, startangle=90)
axes[1].set_title('Event Distribution')

# Plot 3: Interaction density heatmap (sample)
sample_R = R_train[:30, :50].toarray()
im = axes[2].imshow(sample_R > 0, cmap='Blues', aspect='auto')
axes[2].set_xlabel('Items (first 50)')
axes[2].set_ylabel('Users (first 30)')
axes[2].set_title(f'Interaction Matrix (density={density:.1f}%)')

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_comparison.png')

In [ ]:
# Sample recommendations for a user
sample_user = all_users[0]
sample_idx = u2idx[sample_user]

als_ids, als_scores = als_model.recommend(sample_idx, R_train[sample_idx], N=10, filter_already_liked_items=True)
print(f'Top-10 ALS recommendations for {sample_user}:')
for rank, (iid, score) in enumerate(zip(als_ids, als_scores), 1):
    item = all_items[iid]
    cat = item_cat_map.get(item, 'unknown')
    print(f'  {rank}. {item} ({cat}) — score: {score:.3f}')